# EduHinglish: Gemma-3-4B-it Fine-Tuning (T4 GPU)
This notebook fine-tunes Google's Gemma-3-4B-it model on the combined Biology and Social Science Hinglish dataset using QLoRA.

> **Runtime**: Runtime → Change runtime type → **T4 GPU**

## 0. Install Dependencies
Pin versions to avoid API-breaking updates mid-run. **Restart the runtime after this cell completes.**

In [1]:
!pip install -q -U \
    "bitsandbytes>=0.43.0" \
    "transformers>=4.47.0" \
    "peft>=0.14.0" \
    "accelerate>=1.0.0" \
    "datasets>=3.0.0" \
    "trl>=0.12.0,<0.20.0" \
    "huggingface_hub"
print("Done. Please RESTART the runtime (Runtime -> Restart session), then run the remaining cells.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 108.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 60.1 MB/s eta 0:00:00
Done. Please RESTART the runtime (Runtime -> Restart session), then run the remaining cells.


## 1. Authentication & Drive Mount
Gemma is a gated model — you must accept its license first.
1. Go to https://huggingface.co/google/gemma-3-4b-it and click **Acknowledge license**.
2. Go to https://huggingface.co/settings/tokens and create a **Read** token.
3. In Colab, open **Secrets** (key icon in left sidebar) → add `HF_TOKEN` → paste your token → enable **Notebook access**.

In [2]:
from google.colab import drive, userdata
from huggingface_hub import login

drive.mount('/content/drive')

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("HuggingFace login successful.")

Mounted at /content/drive
HuggingFace login successful.


## 2. Load and Prepare Dataset

In [3]:
import json
import random
from datasets import Dataset

base_dir = "/content/drive/MyDrive/EduHinglish-ollama"

try:
    with open(f"{base_dir}/data/bio_9_10.json", "r") as f:
        bio_data = json.load(f)
    with open(f"{base_dir}/data/ss9.json", "r") as f:
        ss9_data = json.load(f)
    with open(f"{base_dir}/data/ss10.json", "r") as f:
        ss10_data = json.load(f)
    print(f"bio_9_10.json: {len(bio_data)} entries")
    print(f"ss9.json     : {len(ss9_data)} entries")
    print(f"ss10.json    : {len(ss10_data)} entries")
except FileNotFoundError as e:
    print(f"ERROR: {e}")
    print(f"Please ensure your files are at {base_dir}/data/")
    bio_data, ss9_data, ss10_data = [], [], []

all_data = bio_data + ss9_data + ss10_data
random.seed(42)
random.shuffle(all_data)
print(f"\nTotal entries loaded: {len(all_data)}")

bio_9_10.json: 7003 entries
ss9.json     : 3668 entries

Total entries loaded: 10671


### Format for Gemma Chat Template

In [4]:
def format_gemma_chat(entry):
    context  = entry.get("ncert_context", "")
    # Use Hinglish question if available, fallback to English
    question = entry.get("question_hinglish") or entry.get("question_english", "")
    intent   = entry.get("intent", "")

    prompt = (
        f"Context:\n{context}\n\n"
        f"Question:\n{question}\n\n"
        f"Intent Type: {intent}\n"
        f"Answer the question based on the context."
    )
    answer = entry.get("answer_hinglish") or entry.get("answer_english", "")

    formatted_text = (
        f"<start_of_turn>user\n{prompt}<end_of_turn>\n"
        f"<start_of_turn>model\n{answer}<end_of_turn>"
    )
    return {"text": formatted_text}

formatted_data = [format_gemma_chat(e) for e in all_data]
dataset = Dataset.from_list(formatted_data)
print(f"Formatted {len(dataset)} entries.")

Formatted 10671 entries.


### Train / Test Split (95 / 5)

In [5]:
split_dataset = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split_dataset["train"]
eval_dataset  = split_dataset["test"]

print(f"Training samples  : {len(train_dataset)}")
print(f"Evaluation samples: {len(eval_dataset)}")
print("\nSample entry:")
print(train_dataset[0]["text"])

Training samples  : 10137
Evaluation samples: 534

Sample entry:
<start_of_turn>user
Context:
A sculpture of figurine from the Sindhu- Viṣhṇu from the Sarasvatī Civilisation 12th century CE 1 – Understanding Social Science Epigraphic Sources Epigraphic Source: A historical source consisting of texts, decrees, or records inscribed on durable materials such as stone, metal plates, or rock surfaces. These inscriptions provide direct evidence about past events, rulers, administration, Fig. 1.6. Brahmi and society. Fig. 1.7. Kannada inscription of Emperor inscription on part of a Krishnadeva Raya at the underground Shiva pillar belonging to the temple, Prasanna Virupaksha Temple, Hampi Gupta period Numismatic Sources Numismatic Source: A historical source comprising coins, currency, or medals.

Question:
What is meant by Understanding Social Science Epigraphic?

Intent Type: definition
Please provide the explanation in Hinglish.<end_of_turn>
<start_of_turn>model
Sindhu Sarasvatī Civilizatio

## 3. Load Tokenizer & Model (QLoRA for T4)
4-bit NF4 quantisation with `fp16` — T4 does **not** support bfloat16.

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "google/gemma-3-4b-it"

# --- Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # required for causal LM training

# --- 4-bit quantisation config ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# --- Load base model ---
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False       # must disable when using gradient checkpointing
model.enable_input_require_grads()   # allows LoRA layers to receive gradients

print(f"Model loaded. dtype={model.dtype}")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Model loaded. dtype=torch.bfloat16


### LoRA Adapters

In [7]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 32,788,480 || all params: 4,332,867,952 || trainable%: 0.7567


## 4. Training
Checkpoints are saved every 100 steps to Drive.

> **API note (trl >= 0.9):** SFT-specific args (`dataset_text_field`, `max_length`, `packing`) live in `SFTConfig`, not in `SFTTrainer`. `max_seq_length` was renamed to `max_length`. Use `processing_class` instead of the deprecated `tokenizer` kwarg.

In [14]:
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

# Correct pattern: Initialize SFTConfig which inherits from TrainingArguments.
# We pass standard TrainingArguments directly, and SFT-specific arguments are set cleanly.
sft_config = SFTConfig(
    output_dir=f"{base_dir}/training/gemma-eduhinglish-v1",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
    fp16=True,
    bf16=False,
    max_steps=500,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    report_to="none",

    # SFT-specific arguments inside SFTConfig
    dataset_text_field="text",
    max_seq_length=1024,
    packing=False,
)

# Create trainer using SFTConfig
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    args=sft_config,
)

print("Starting Training...")
trainer.train()

Adding EOS to train dataset:   0%|          | 0/10137 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10137 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/10137 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/534 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/534 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/534 [00:00<?, ? examples/s]

Starting Training...


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
100,1.450778,1.443098
200,1.226373,1.196548
300,1.100128,0.998211
400,0.752638,0.882175
500,0.848612,0.850843


TrainOutput(global_step=500, training_loss=1.2199053497314454, metrics={'train_runtime': 5509.7861, 'train_samples_per_second': 0.726, 'train_steps_per_second': 0.091, 'total_flos': 3.291220356828768e+16, 'train_loss': 1.2199053497314454})

## 5. Save Adapter & Test Inference

In [15]:
adapter_path = f"{base_dir}/training/gemma-eduhinglish-adapter"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved to: {adapter_path}")

Adapter saved to: /content/drive/MyDrive/EduHinglish-ollama/training/gemma-eduhinglish-adapter


In [16]:
model.eval()

def generate_response(prompt_text, max_new_tokens=200):
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    input_len = inputs["input_ids"].shape[1]
    # Decode only newly generated tokens (skip echoed prompt)
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

test_entry  = eval_dataset[0]["text"]
prompt_only = test_entry.split("<start_of_turn>model")[0] + "<start_of_turn>model\n"

print("--- PROMPT ---")
print(prompt_only)
print("\n--- GENERATED HINGLISH RESPONSE ---")
print(generate_response(prompt_only))

--- PROMPT ---
<start_of_turn>user
Context:
These cells proliferate and make large numbers of cells. From this mass of cells, different cells undergo changes to become various cell types and tissues. These changes -27 take place in an organised sequence referred to as development. However, regeneration is not the same as reproduction, since most organisms would not normally depend on being cut up to be able to reproduce. 7.2.4 Budding Organisms such as Hydra use regenerative cells for reproduction in the process of budding. In Hydra, a bud Regeneration in Planaria develops as an outgrowth due to repeated cell division at one specific site ().

Question:
What are the causes or factors responsible for Budding Organisms?

Intent Type: explain_concept
Please provide the explanation in Hinglish.<end_of_turn>
<start_of_turn>model


--- GENERATED HINGLISH RESPONSE ---
Yeh cells zyada zyada ho jaate hain aur ek bari number of cells banate hain. Is cells ke mass se, alag-alag cells different ce

MODEL EVAL

In [17]:
!pip install -q rouge_score sacrebleu

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 7.4 MB/s eta 0:00:00


In [18]:
import re, math, torch
import torch.nn.functional as F
from tqdm.auto import tqdm

MODEL_TAG = "<start_of_turn>model\n"

def split_entry(text):
    prompt_part, answer = text.split(MODEL_TAG, 1)
    return prompt_part + MODEL_TAG, answer.replace("<end_of_turn>", "").strip()

@torch.no_grad()
def eval_token_metrics(ds, max_samples=None, max_length=1024):
    model.eval()
    n = len(ds) if max_samples is None else min(max_samples, len(ds))
    tot_loss = tot_correct = tot_tokens = 0
    for i in tqdm(range(n)):
        text = ds[i]["text"]
        prompt_part, _ = split_entry(text)
        p_len = len(tokenizer(prompt_part)["input_ids"])
        enc = tokenizer(text, return_tensors="pt", truncation=True,
                        max_length=max_length).to(model.device)
        ids = enc["input_ids"]
        if ids.shape[1] <= p_len:
            continue
        logits = model(**enc, use_cache=False).logits[0, :-1].float()
        targets = ids[0, 1:]
        mask = torch.arange(targets.shape[0], device=targets.device) >= (p_len - 1)
        tot_loss    += F.cross_entropy(logits[mask], targets[mask], reduction="sum").item()
        tot_correct += (logits[mask].argmax(-1) == targets[mask]).sum().item()
        tot_tokens  += mask.sum().item()
    loss = tot_loss / tot_tokens
    return {"answer_loss": loss, "perplexity": math.exp(loss),
            "token_accuracy": tot_correct / tot_tokens, "tokens": tot_tokens}

m = eval_token_metrics(eval_dataset)
print(f"Answer loss    : {m['answer_loss']:.4f}")
print(f"Perplexity     : {m['perplexity']:.2f}")
print(f"Token accuracy : {m['token_accuracy']*100:.2f}%  ({m['tokens']} tokens)")

  0%|          | 0/534 [00:00<?, ?it/s]

Answer loss    : 0.7830
Perplexity     : 2.19
Token accuracy : 80.06%  (66080 tokens)


In [19]:
end_turn_id = tokenizer.convert_tokens_to_ids("<end_of_turn>")

def generate(prompt_text, max_new_tokens=300, greedy=True):
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    model.config.use_cache = True
    kwargs = dict(max_new_tokens=max_new_tokens, use_cache=True,
                  pad_token_id=tokenizer.pad_token_id,
                  eos_token_id=[tokenizer.eos_token_id, end_turn_id])
    if greedy:
        kwargs["do_sample"] = False
    else:
        kwargs.update(do_sample=True, temperature=0.7, top_p=0.9)
    with torch.no_grad():
        out = model.generate(**inputs, **kwargs)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def build_prompt(question, context="", intent="explain"):
    user = (f"Context:\n{context}\n\nQuestion:\n{question}\n\n"
            f"Intent Type: {intent}\nPlease provide the explanation in Hinglish.")
    return f"<start_of_turn>user\n{user}<end_of_turn>\n{MODEL_TAG}"

print("Valid intent labels in your data:", sorted({str(e.get('intent')) for e in all_data}))

Valid intent labels in your data: ['analytical', 'application_reallife', 'assertion_reason', 'compare_concepts', 'definition', 'describe_function', 'explain_concept', 'fill_blank', 'formula_application', 'give_example', 'list_enumerate', 'mcq', 'name_the', 'numerical_problem', 'one_word', 'reaction_equation', 'short_answer', 'true_false']


In [20]:
from rouge_score import rouge_scorer
import sacrebleu, numpy as np

class UniTok:
    def tokenize(self, text):
        return re.findall(r"\w+", text.lower())

scorer = rouge_scorer.RougeScorer(["rougeL"], tokenizer=UniTok())
deva = re.compile(r"[\u0900-\u097F]")

def devanagari_ratio(t):
    letters = [c for c in t if c.isalpha()]
    return sum(bool(deva.match(c)) for c in letters) / max(len(letters), 1)

def run_generation_eval(ds, n=20, use_base=False):
    preds, refs = [], []
    for i in tqdm(range(n), desc="base" if use_base else "finetuned"):
        prompt, ref = split_entry(ds[i]["text"])
        if use_base:
            with model.disable_adapter():
                pred = generate(prompt)
        else:
            pred = generate(prompt)
        preds.append(pred); refs.append(ref)
    rl = np.mean([scorer.score(r, p)["rougeL"].fmeasure for r, p in zip(refs, preds)])
    chrf = sacrebleu.corpus_chrf(preds, [refs]).score
    return {"rougeL": rl, "chrF": chrf,
            "devanagari_ratio": np.mean([devanagari_ratio(p) for p in preds]),
            "avg_len_words": np.mean([len(p.split()) for p in preds]),
            "preds": preds, "refs": refs}

N = 20
ft   = run_generation_eval(eval_dataset, N)
base = run_generation_eval(eval_dataset, N, use_base=True)

print(f"\n{'metric':<18}{'base':>10}{'fine-tuned':>12}")
for k in ["rougeL", "chrF", "devanagari_ratio", "avg_len_words"]:
    print(f"{k:<18}{base[k]:>10.3f}{ft[k]:>12.3f}")
print(f"{'reference':<18}{'':>10}{np.mean([devanagari_ratio(r) for r in ft['refs']]):>12.3f}  (devanagari ratio of reference answers)")

finetuned:   0%|          | 0/20 [00:00<?, ?it/s]

base:   0%|          | 0/20 [00:00<?, ?it/s]


metric                  base  fine-tuned
rougeL                 0.077       0.548
chrF                  26.427      61.889
devanagari_ratio       0.121       0.000
avg_len_words        157.100      89.700
reference                          0.081  (devanagari ratio of reference answers)


In [22]:
tests = [
    # (question, context, intent, [keywords that a correct answer must contain])
    ("How does Pharaoh Government Officials work? Describe step by step.", "", "explain", [""]),
    # ("", "", "explain", [""]),
    # add more here...
]

def keyword_hits(answer, kws):
    a = answer.lower()
    return sum(k.lower() in a for k in kws) / max(len(kws), 1)

scores = {"base": [], "finetuned": []}
for q, ctx, intent, kws in tests:
    prompt = build_prompt(q, ctx, intent)
    with model.disable_adapter():
        b = generate(prompt)
    f = generate(prompt)
    scores["base"].append(keyword_hits(b, kws)); scores["finetuned"].append(keyword_hits(f, kws))
    print("="*80, f"\nQ: {q}  [{intent}]")
    print(f"\n--- BASE (kw {scores['base'][-1]:.0%}) ---\n{b}")
    print(f"\n--- FINE-TUNED (kw {scores['finetuned'][-1]:.0%}) ---\n{f}")

print("\nKeyword accuracy  base: %.1f%%  fine-tuned: %.1f%%" % (
    100*np.mean(scores["base"]), 100*np.mean(scores["finetuned"])))

Q: How does Pharaoh Government Officials work? Describe step by step.  [explain]

--- BASE (kw 100%) ---
Okay, let's break down how the Pharaoh's government officials worked, step-by-step, in a way that's easy to understand – in Hinglish (Hindi & English mix)!

**Pharaoh’s Government: A Step-by-Step Look**

The Pharaoh was basically the absolute ruler, like a king but with a huge claim to being a god.  His power wasn't just about fighting; it was about *managing* everything.  So, he needed a whole team of officials to do that. Here’s how it worked:

**1. The Pharaoh Himself (The Top Boss):**

* **Decision Maker:** The Pharaoh made *all* the big decisions – laws, war, building projects, religious matters. He listened to his advisors, but ultimately, the final say was always his.
* **Supreme Judge:** He could also act as a judge, settling disputes and punishing wrongdoers.
* **Religious Leader:** He was considered the intermediary between the gods and the people, overseeing religious cer